# Diagnostic — which benefits are extracting vs failing

Run this against your `compare_payload_vs_output` results. It tells you:
- Which benefit IDs are extracting across plans (filtering works)
- Which benefit IDs are returning zero everywhere (broken)
- For broken benefits: do the input PBP rows actually contain matchable categories?

This isolates **prompt/extraction problems** from **input-data missing** from **filter-not-matching**.


In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd

# ── Same paths as before ──────────────────────────────────────────────────────
PAYLOAD_PATH = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')
OUTPUT_PATH  = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_output.json')

# Load both
payload_raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
payload_rows = payload_raw['pbp'] if isinstance(payload_raw, dict) and 'pbp' in payload_raw else payload_raw

output_rows = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
if isinstance(output_rows, dict) and 'results' in output_rows:
    output_rows = output_rows['results']

print(f'Payload: {len(payload_rows):,} rows | Output: {len(output_rows):,} rows')

# Mirror BENEFIT_TARGETS from build_benefits.py
BENEFIT_TARGETS = [
    ('600',  'Monthly Premium',            [],             ['Monthly Premium', 'Plan Premium']),
    ('610',  'Health Plan Deductible',     [],             ['Health Plan Deductible', 'Medical Deductible']),
    ('611',  'Drug Deductible',            [],             ['Drug Deductible', 'Rx Deductible', 'Enter Deductible Amount']),
    ('614',  'Health Monthly Premium',     [],             ['Health Monthly Premium', 'Health Premium']),
    ('615',  'Drug Monthly Premium',       [],             ['Drug Monthly Premium', 'Rx Premium', 'Part D Premium']),
    ('616',  'Part B Premium Reduction',   [],             ['Part B Premium', 'Part B Reduction', 'Part B giveback']),
    ('620',  'Out-of-Pocket Spending',     [],             ['MOOP', 'Max Enrollee Cost', 'Out of Pocket']),
    ('700',  'Tier Names',                 [],             ['Rx Tier', 'Formulary Tier', 'Tier Names']),
    ('710',  'Initial Coverage',           [],             ['Initial Coverage Phase', 'Rx Setup']),
    ('711',  'Retail Pharmacy',            [],             ['Retail Pharmacy', 'Initial Coverage Phase']),
    ('730',  'Catastrophic Coverage',      [],             ['Catastrophic Coverage']),
    ('740',  'Formulary Exception',        [],             ['Formulary Exception']),
    ('755',  'Preferred Mail Order',       [],             ['Preferred Mail Order']),
    ('760',  'Standard Mail Order',        [],             ['Standard Mail Order']),
    ('800',  'Inpatient Hospital',         ['1a'],         ['Inpatient Hospital']),
    ('810',  'Inpatient Mental Health',    ['1b'],         ['Inpatient Mental Health', 'Inpatient Psychiatric']),
    ('820',  'Skilled Nursing Facility',   ['2'],          ['Skilled Nursing', 'SNF']),
    ('900',  'PCP',                        ['7a'],         ['Primary Care', 'PCP']),
    ('910',  'Specialist',                 ['7b'],         ['Specialist', 'Specialty Care']),
    ('911',  'Telehealth',                 ['7d'],         ['Telehealth', 'Telemedicine', 'Virtual']),
    ('920',  'Chiropractic',               ['8'],          ['Chiropractic']),
    ('930',  'Podiatry',                   ['9a'],         ['Podiatry']),
    ('940',  'Outpatient Mental Health',   ['4a'],         ['Outpatient Mental Health']),
    ('950',  'Outpatient Substance Abuse', ['4b'],         ['Substance Abuse']),
    ('960',  'Outpatient Surgery',         ['4c'],         ['Outpatient Surgery', 'Outpatient Services', 'Ambulatory Surgical']),
    ('970',  'Ambulance',                  ['5'],          ['Ambulance']),
    ('981',  'Emergency',                  ['6a'],         ['Emergency']),
    ('982',  'Urgent Care',                ['6b'],         ['Urgent Care', 'Urgently Needed']),
    ('990',  'Outpatient Rehab',           ['10'],         ['Outpatient Rehabilitation', 'Physical Therapy', 'Occupational Therapy']),
    ('1000', 'DME',                        ['11a'],        ['Durable Medical Equipment', 'DME']),
    ('1020', 'Diabetes',                   ['11c'],        ['Diabetic', 'Diabetes']),
    ('1030', 'Diagnostic/Lab/Radiology',   ['3'],          ['Diagnostic', 'X-Ray', 'Lab', 'Radiology']),
    ('1050', 'Fitness',                    ['13b'],        ['Fitness', 'SilverSneakers']),
    ('1060', 'Meals',                      ['13c'],        ['Meals']),
    ('1200', 'Kidney',                     ['12'],         ['Kidney', 'Renal', 'Dialysis']),
    ('1300', 'Preventive Dental',          ['16a'],        ['Preventive Dental']),
    ('1301', 'Comprehensive Dental',       ['16b'],        ['Comprehensive Dental']),
    ('1400', 'Hearing',                    ['18a', '18b'], ['Hearing Exam', 'Hearing Aid']),
    ('1500', 'Vision',                     ['17a', '17b'], ['Vision', 'Eyewear', 'Eyeglasses']),
    ('1610', 'Prosthetics',                ['11b'],        ['Prosthetic']),
    ('1700', 'Preventive',                 ['14a', '14b', '14c'], ['Preventive', 'Wellness Visit']),
    ('1800', 'Transportation',             ['15'],         ['Transportation']),
    ('1900', 'Acupuncture',                ['13a'],        ['Acupuncture']),
    ('2100', 'OTC',                        ['13e'],        ['Over-the-Counter', 'OTC']),
    ('2110', 'Plan Rating',                [],             ['Plan Rating', 'Star Rating']),
    ('2111', 'Health Plan Rating',         [],             ['Health Plan Rating']),
    ('2112', 'Drug Plan Rating',           [],             ['Drug Plan Rating']),
]
print(f'Benefit targets: {len(BENEFIT_TARGETS)}')


## 1. Output: rows per benefit ID, across all plans

If a benefit has 0 across all 67 plans, something is structurally wrong with that benefit's extraction. If it has rows in some plans but not others, that's plan-specific data variation (likely fine).

In [ ]:
out_df = pd.DataFrame(output_rows)
out_df['benefitid'] = out_df['benefitid'].astype(str)

bid_counts = out_df.groupby('benefitid').size()
plans_with_bid = out_df.groupby('benefitid')['planid'].nunique()

# Build summary
expected_ids = [t[0] for t in BENEFIT_TARGETS]
expected_names = {t[0]: t[1] for t in BENEFIT_TARGETS}
n_plans = out_df['planid'].nunique()

summary = []
for bid in expected_ids:
    total_rows = int(bid_counts.get(bid, 0))
    plans_seen = int(plans_with_bid.get(bid, 0))
    summary.append({
        'benefitid':   bid,
        'name':        expected_names.get(bid, ''),
        'total_rows':  total_rows,
        'plans_w_data': plans_seen,
        'pct_plans':   f'{100*plans_seen/n_plans:.0f}%',
        'status':      'OK' if plans_seen > 0 else 'ZERO',
    })
sum_df = pd.DataFrame(summary).sort_values('total_rows', ascending=False)
display(sum_df)

zero_benefits = sum_df[sum_df['status']=='ZERO']
print(f'\nBenefits with ZERO rows across all {n_plans} plans: {len(zero_benefits)}')
for _, row in zero_benefits.iterrows():
    print(f'   {row["benefitid"]} {row["name"]}')


## 2. For each ZERO benefit — does the input data even contain matchable rows?

Mimic the filtering logic from `build_benefits.py`. If the input has zero matchable rows, the LLM call was correctly skipped — the data genuinely doesn't have this benefit. If the input HAS rows but output is empty, the filtering matched but the LLM failed.

In [ ]:
# Mirror _row_matches_target exactly
def row_matches_target(row, section_codes, keywords):
    category = (row.get('category') or '').lower()
    for code in section_codes:
        if f'({code})' in category or f'({code}1)' in category or f'({code}2)' in category:
            return True
    for kw in keywords:
        if kw.lower() in category:
            return True
    return False

# For each zero benefit, scan the entire payload for matches
print(f'Diagnosing {len(zero_benefits)} zero-output benefits:\n')
print(f'{"bid":>5} {"name":<28} {"matched_rows":>12} {"distinct_cats":>13}  diagnosis')
print('-' * 100)

for bid in zero_benefits['benefitid']:
    target = next(t for t in BENEFIT_TARGETS if t[0] == bid)
    _, name, codes, keywords = target

    matched_rows = [r for r in payload_rows if row_matches_target(r, codes, keywords)]
    distinct_cats = sorted({r.get('category','') for r in matched_rows})

    if not matched_rows:
        diag = 'INPUT_HAS_NONE  (filter found nothing — data may use different category strings)'
    else:
        diag = f'LLM_FAILED      (filter found {len(matched_rows)} rows but LLM returned 0)'

    print(f'{bid:>5} {name:<28} {len(matched_rows):>12} {len(distinct_cats):>13}  {diag}')
    # Show up to 3 example categories so we can see what the actual data looks like
    for c in distinct_cats[:3]:
        print(f'        sample category: {c[:90]}')


## 3. What category strings DOES the input actually use?

Most useful diagnostic if benefits show INPUT_HAS_NONE: dump all unique categories in the payload so you can spot the actual section codes and rebuild the BENEFIT_TARGETS mapping.

In [ ]:
# Extract section codes like (7a), (13a), (18b1) from every category
section_pattern = re.compile(r'\((\d+[a-z]?\d?)\)')

cat_counter = Counter()
section_counter = Counter()
for r in payload_rows:
    cat = (r.get('category') or '').strip()
    if cat:
        cat_counter[cat] += 1
        for m in section_pattern.finditer(cat):
            section_counter[m.group(1)] += 1

print(f'Distinct categories in payload: {len(cat_counter)}')
print(f'Distinct section codes found:   {len(section_counter)}\n')

print('Most common section codes:')
for code, n in section_counter.most_common(40):
    print(f'   ({code})  {n:>6,} rows')


### Top 50 unique categories (sorted by frequency)

In [ ]:
print(f'{"count":>6}  category')
print('-' * 100)
for cat, n in cat_counter.most_common(50):
    print(f'{n:>6,}  {cat[:90]}')


## 4. The headline diagnosis

If most ZERO benefits show INPUT_HAS_NONE → the BENEFIT_TARGETS section codes and keywords don't match how this carrier's PBP data is structured. Need to update the mapping.

If most ZERO benefits show LLM_FAILED → the input filter works, but the LLM call is returning empty/invalid responses. Need to look at the Flask logs or tighten the prompt.

In [ ]:
# Final summary
n_input_none = 0
n_llm_failed = 0
for bid in zero_benefits['benefitid']:
    target = next(t for t in BENEFIT_TARGETS if t[0] == bid)
    _, _, codes, keywords = target
    matched = sum(1 for r in payload_rows if row_matches_target(r, codes, keywords))
    if matched == 0:
        n_input_none += 1
    else:
        n_llm_failed += 1

total_zero = len(zero_benefits)
total_ok   = len(expected_ids) - total_zero
print(f'Out of {len(expected_ids)} expected benefits:')
print(f'  ✓ {total_ok:>3} extracting data')
print(f'  ✗ {n_input_none:>3} ZERO because filter found no matching rows (data shape issue)')
print(f'  ✗ {n_llm_failed:>3} ZERO despite filter matching rows (LLM/prompt issue)')

if n_input_none > n_llm_failed:
    print('\n→ Next step: update BENEFIT_TARGETS section codes/keywords to match this data.')
else:
    print('\n→ Next step: inspect Flask logs for failed per-benefit LLM calls. '
          'Likely a prompt or parsing issue.')
